In [3]:
import os
import pandas as pd
import torch
from PIL import Image
from open_flamingo import create_model_and_transforms
from huggingface_hub import hf_hub_download
from tqdm import tqdm

# -- 1. 설정 (Configuration) --
# 데이터 경로 설정
TEST_CSV_PATH = '/data/2_data_server/cv-07/dice/2025_samsung_challenge/data/dev_test.csv'
IMAGE_BASE_PATH = '/data/2_data_server/cv-07/dice/2025_samsung_challenge/data'
SUBMISSION_CSV_PATH = './submission_final.csv' # 최종 결과를 저장할 경로

# 디바이스 설정 (GPU 우선 사용)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {DEVICE}")

# -- 2. 모델 및 전처리기 로드 --
print("Loading OpenFlamingo model and transforms...")
# (이 부분은 이전 코드와 동일합니다)
model, image_processor, tokenizer = create_model_and_transforms(
    clip_vision_encoder_path="ViT-L-14",
    clip_vision_encoder_pretrained="openai",
    lang_encoder_path="anas-awadalla/mpt-1b-redpajama-200b",
    tokenizer_path="anas-awadalla/mpt-1b-redpajama-200b",
    cross_attn_every_n_layers=1,
)
checkpoint_path = hf_hub_download("openflamingo/OpenFlamingo-3B-vitl-mpt1b", "checkpoint.pt")
model.load_state_dict(torch.load(checkpoint_path), strict=False)
model.to(DEVICE)
model.eval()
print("Model loading complete.")

# -- 3. 데이터 로드 --
try:
    df_test = pd.read_csv(TEST_CSV_PATH)
    print(f"Test data loaded. Total questions: {len(df_test)}")
except FileNotFoundError:
    print(f"Error: {TEST_CSV_PATH} 파일을 찾을 수 없습니다. 경로를 확인해주세요.")
    exit()

# -- 4. 추론 실행 (결과는 숫자로 저장) --
results = []
tokenizer.padding_side = "left"

for idx, row in tqdm(df_test.iterrows(), total=len(df_test), desc="OpenFlamingo Inference (1,2,3,4)"):
    
    img_path = os.path.join(IMAGE_BASE_PATH, row['img_path'].lstrip('./'))
    try:
        image = Image.open(img_path).convert("RGB")
    except FileNotFoundError:
        print(f"Error: Image not found at {img_path}")
        results.append({'ID': row['ID'], 'answer': -1}) # 에러는 -1로 표기
        continue

    question = row['Question']
    choices = [row['A'], row['B'], row['C'], row['D']]
    
    choice_losses = []
    vision_x = image_processor(image).unsqueeze(0).unsqueeze(0).unsqueeze(0).to(DEVICE)
    
    with torch.no_grad():
        for choice in choices:
            prompt = f"<image>Question: {question} Answer: {choice}"
            lang_x = tokenizer([prompt], return_tensors="pt").to(DEVICE)
            outputs = model(
                vision_x=vision_x,
                lang_x=lang_x["input_ids"],
                attention_mask=lang_x["attention_mask"],
                labels=lang_x["input_ids"]
            )
            loss = outputs.loss
            choice_losses.append(loss.item())

    # loss가 가장 낮은 선택지의 인덱스(0, 1, 2, 3)를 찾습니다.
    predicted_index = choice_losses.index(min(choice_losses))
    # 제출을 위해 1-based index (1, 2, 3, 4)로 변환합니다.
    predicted_answer = predicted_index + 1
    
    results.append({'ID': row['ID'], 'answer': predicted_answer})

# -- 5. 후처리 (Post-processing) --

# 먼저, 숫자 결과로 데이터프레임을 생성합니다.
submission_df = pd.DataFrame(results)

print("\n--- 숫자 답변 미리보기 (변환 전) ---")
print(submission_df.head())
print("-----------------------------------")

# 숫자(1,2,3,4)를 문자(A,B,C,D)로 변환하기 위한 매핑 딕셔너리
answer_map = {1: 'A', 2: 'B', 3: 'C', 4: 'D', -1: 'Error'}

# pandas의 map 함수를 사용하여 'answer' 컬럼 전체를 한 번에 변환합니다.
submission_df['answer'] = submission_df['answer'].map(answer_map)

print("\n후처리 완료: 숫자 답변을 A, B, C, D 문자로 변환했습니다.")

# -- 6. 최종 결과 저장 및 출력 --

submission_df.to_csv(SUBMISSION_CSV_PATH, index=False)

print(f"\n최종 결과가 {SUBMISSION_CSV_PATH} 파일에 저장되었습니다.")
print("\n--- 최종 제출 파일 미리보기 (변환 후) ---")
print(submission_df.head())
print("----------------------------------------")

Using device: cuda
Loading OpenFlamingo model and transforms...


/data/2_data_server/cv-07/anaconda3/envs/nlp/lib/python3.11/site-packages/open_clip/factory.py:388: UserWarning: These pretrained weights were trained with QuickGELU activation but the model config does not have that enabled. Consider using a model config with a "-quickgelu" suffix or enable with a flag.
  warnings.warn(
/home/cv-07/.cache/huggingface/modules/transformers_modules/anas-awadalla/mpt-1b-redpajama-200b/50d6bc94e17812873f39c36c5f815263fa71fb80/attention.py:289: UserWarning: Using `attn_impl: torch`. If your model does not use `alibi` or `prefix_lm` we recommend using `attn_impl: flash` otherwise we recommend using `attn_impl: triton`.
  warnings.warn(


You are using config.init_device='cpu', but you can also use config.init_device="meta" with Composer + FSDP for fast initialization.
Flamingo model initialized with 1046992944 trainable parameters
Model loading complete.
Test data loaded. Total questions: 60


OpenFlamingo Inference (1,2,3,4): 100%|██████████| 60/60 [00:13<00:00,  4.31it/s]


--- 숫자 답변 미리보기 (변환 전) ---
         ID  answer
0  TEST_000       1
1  TEST_001       1
2  TEST_002       2
3  TEST_003       3
4  TEST_004       2
-----------------------------------

후처리 완료: 숫자 답변을 A, B, C, D 문자로 변환했습니다.

최종 결과가 ./submission_final.csv 파일에 저장되었습니다.

--- 최종 제출 파일 미리보기 (변환 후) ---
         ID answer
0  TEST_000      A
1  TEST_001      A
2  TEST_002      B
3  TEST_003      C
4  TEST_004      B
----------------------------------------


In [ ]:
# 전체 파라미터와 학습 가능한 파라미터 수를 계산
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"전체 파라미터: {total_params:,}개 ({total_params / 1_000_000:.2f}M)")
print(f"학습 가능한 파라미터: {trainable_params:,}개 ({trainable_params / 1_000_000:.2f}M)")

전체 파라미터: 2,559,117,360개 (2559.12M)
학습 가능한 파라미터: 1,046,992,944개 (1046.99M)
